Import Libraries And Load The Clean Dataset

In [ ]:
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv('../notebooks/mental_health_dataset_cleaned.csv')
df.head()


Drop NaNs from clean_text

In [2]:
df = df.dropna(subset=['clean_text'])
print(df['clean_text'].isnull().sum())  # should now be 0

0


Selecting Only What Is Needed and Initialize TF-IDF

In [3]:
X = df['clean_text']
y = df['status'] 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
vectorizer = TfidfVectorizer(
    min_df=5,
    max_df=0.9,
    max_features=15000
)

Learn All Unique Words And Convert Training Text To Number

In [4]:
X_train_tfidf = vectorizer.fit_transform(X_train)

Transform Test Text to Number 

In [5]:
X_test_tfidf = vectorizer.transform(X_test)

Check Shape

In [6]:
X_train_tfidf.shape

(39623, 8154)

Show Some Feature Names

In [ ]:
print(vectorizer.get_feature_names_out()[:50])

Show the actual numbers generated for each word:

In [ ]:
df_tfidf = pd.DataFrame(X_train_tfidf.toarray(), columns=vectorizer.get_feature_names_out())
print(df_tfidf.head())

See non-zero words for first post

In [ ]:
row = df_tfidf.iloc[0]
print(row[row > 0])

 LOGISTIC REGRESSION 

In [10]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)
lr_predictions = lr_model.predict(X_test_tfidf)

SVM(Support Vector Machine)

In [11]:
svm_model = LinearSVC(max_iter=1000, random_state=42)
svm_model.fit(X_train_tfidf, y_train)
svm_predictions = svm_model.predict(X_test_tfidf)

In [12]:
def predict_status(text):
    vectorized = vectorizer.transform([text])
    prediction = lr_model.predict(vectorized)
    return prediction[0]

# Test it
user_input = input("Enter a post: ")
print("Predicted Status:", predict_status(user_input))

Predicted Status: Normal


Import libraries needed for Evaluation

In [13]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# ======= CLASSIFICATION REPORT =======
print("LOGISTIC REGRESSION REPORT")
print(classification_report(y_test, lr_predictions))

In [ ]:
print("SVM REPORT")
print(classification_report(y_test, svm_predictions))

In [ ]:
# ======= CONFUSION MATRIX - LOGISTIC REGRESSION =======
plt.figure(figsize=(8,6))
sns.heatmap(confusion_matrix(y_test, lr_predictions), 
            annot=True, fmt='d', cmap='Blues',
            xticklabels=['Anxiety','Depression','Normal','Suicidal'],
            yticklabels=['Anxiety','Depression','Normal','Suicidal'])
plt.title('Logistic Regression Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
# ======= CONFUSION MATRIX - SVM =======
plt.figure(figsize=(8,6))
sns.heatmap(confusion_matrix(y_test, svm_predictions), 
            annot=True, fmt='d', cmap='Reds',
            xticklabels=['Anxiety','Depression','Normal','Suicidal'],
            yticklabels=['Anxiety','Depression','Normal','Suicidal'])
plt.title('SVM Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [22]:
import joblib

joblib.dump(lr_model, '../models/lr_model.pkl')
joblib.dump(svm_model, '../models/svm_model.pkl')
joblib.dump(vectorizer, '../models/vectorizer.pkl')

print("Models saved successfully ✅")

Models saved successfully ✅
